In [ ]:
!pip install -q ultralytics moviepy transformers torch torchvision pillow ffmpeg-python

In [ ]:
import os
import cv2
import torch
import ffmpeg
from moviepy.editor import VideoFileClip
from ultralytics import YOLO
from transformers import AutoProcessor, AutoModelForImageClassification

# ==============================
# إعداد المسارات
# ==============================
video_path = "/content/Clint Eastwood Spends The Night With A Lady  High Plains Drifter - Critic Picks (720p, h264, youtube).mp4"      # ← غيري ده للفيديو عندك
audio_path = "/content/audio.wav"
safe_video = "/content/blurred_video.mp4"
final_video = "/content/final_output.mp4"

# ==============================
# استخراج الصوت من الفيديو
# ==============================
clip = VideoFileClip(video_path)
clip.audio.write_audiofile(audio_path)
clip.close()

# ==============================
# تحميل النموذجين (YOLO + NSFW)
# ==============================
print("⏳ تحميل النماذج...")
model_yolo = YOLO("yolov8n.pt")  # لاكتشاف الأشخاص
model_nsfw = AutoModelForImageClassification.from_pretrained("Falconsai/nsfw_image_detection")
processor = AutoProcessor.from_pretrained("Falconsai/nsfw_image_detection")
print("✅ تم تحميل النماذج.")

# ==============================
# معالجة الفيديو
# ==============================
cap = cv2.VideoCapture(video_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(safe_video, fourcc, fps, (w, h))

print(f"🎞️ تحليل الفيديو ({frames} فريم)")

frame_num = 0
blurred_frames = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_num += 1

    # تقليص الصورة لتسريع التحليل
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    inputs = processor(images=img_rgb, return_tensors="pt")
    outputs = model_nsfw(**inputs)
    predicted_class = outputs.logits.softmax(dim=1).argmax().item()
    label = model_nsfw.config.id2label[predicted_class]

    if label in ["porn", "sexy"]:
        # لو المشهد غير لائق → تضبيب كل الفريم
        frame = cv2.GaussianBlur(frame, (55, 55), 0)
        blurred_frames += 1

    out.write(frame)
    if frame_num % 50 == 0:
        print(f"⏳ تمت معالجة {frame_num}/{frames} فريم...")

cap.release()
out.release()
print(f"✅ تم حفظ الفيديو بعد التضبيب: {safe_video}")
print(f"🔢 عدد الفريمات المضببة: {blurred_frames}/{frame_num}")

# ==============================
# دمج الصوت
# ==============================
print("🎧 دمج الصوت مع الفيديو النهائي...")
(
    ffmpeg
    .input(safe_video)
    .input(audio_path)
    .output(final_video, vcodec='libx264', acodec='aac', strict='experimental')
    .overwrite_output()
    .run(quiet=True)
)

print("✅ جاهز! الفيديو النهائي:", final_video)


MoviePy - Writing audio in /content/audio.wav


MoviePy - Done.
⏳ تحميل النماذج...


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ تم تحميل النماذج.
🎞️ تحليل الفيديو (7733 فريم)
⏳ تمت معالجة 50/7733 فريم...
⏳ تمت معالجة 100/7733 فريم...
⏳ تمت معالجة 150/7733 فريم...
⏳ تمت معالجة 200/7733 فريم...
⏳ تمت معالجة 250/7733 فريم...
⏳ تمت معالجة 300/7733 فريم...
⏳ تمت معالجة 350/7733 فريم...
⏳ تمت معالجة 400/7733 فريم...
⏳ تمت معالجة 450/7733 فريم...
⏳ تمت معالجة 500/7733 فريم...
⏳ تمت معالجة 550/7733 فريم...
⏳ تمت معالجة 600/7733 فريم...
⏳ تمت معالجة 650/7733 فريم...
⏳ تمت معالجة 700/7733 فريم...
⏳ تمت معالجة 750/7733 فريم...
⏳ تمت معالجة 800/7733 فريم...
⏳ تمت معالجة 850/7733 فريم...
⏳ تمت معالجة 900/7733 فريم...
⏳ تمت معالجة 950/7733 فريم...
⏳ تمت معالجة 1000/7733 فريم...
⏳ تمت معالجة 1050/7733 فريم...
⏳ تمت معالجة 1100/7733 فريم...
⏳ تمت معالجة 1150/7733 فريم...
⏳ تمت معالجة 1200/7733 فريم...
⏳ تمت معالجة 1250/7733 فريم...
⏳ تمت معالجة 1300/7733 فريم...
⏳ تمت معالجة 1350/7733 فريم...
⏳ تمت معالجة 1400/7733 فريم...
⏳ تمت معالجة 1450/7733 فريم...
⏳ تمت معالجة 1500/7733 فريم...
⏳ تمت معالجة 1550/7733 فريم...
⏳ تمت معال

AttributeError: 'FilterableStream' object has no attribute 'input'

In [ ]:
import ffmpeg

try:
    print("🎧 جارٍ دمج الصوت مع الفيديو...")
    input_video = ffmpeg.input(safe_video)
    input_audio = ffmpeg.input(audio_path)

    (
        ffmpeg
        .output(input_video, input_audio, final_video, vcodec='libx264', acodec='aac', strict='experimental')
        .overwrite_output()
        .run(quiet=True)
    )

    print(f"✅ تم حفظ الفيديو النهائي مع الصوت: {final_video}")
except Exception as e:
    print("❌ حدث خطأ أثناء دمج الصوت والفيديو:", e)


🎧 جارٍ دمج الصوت مع الفيديو...
✅ تم حفظ الفيديو النهائي مع الصوت: /content/final_output.mp4


In [ ]:
import cv2
import numpy as np
import moviepy.editor as mp
from ultralytics import YOLO
from faster_whisper import WhisperModel
from better_profanity import profanity
import torch
import os

# ==========================================================
# تحميل نموذج YOLO
# ==========================================================
model = YOLO("yolov8n.pt")

# ==========================================================
# تحميل نموذج Whisper
# ==========================================================
whisper_model = WhisperModel("base", device="cuda" if torch.cuda.is_available() else "cpu")

# ==========================================================
# تحميل قاموس الكلمات غير اللائقة
# ==========================================================
profanity.load_censor_words()

video_path = "/content/video150.mp4"
output_path = "output_blurred_skin.mp4"

# ==========================================================
# دالة لتحديد مناطق الجلد (Skin Detection)
# ==========================================================
def detect_skin(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 48, 80], dtype=np.uint8)
    upper = np.array([20, 255, 255], dtype=np.uint8)
    skin_mask = cv2.inRange(hsv, lower, upper)
    skin_mask = cv2.GaussianBlur(skin_mask, (3, 3), 0)
    return skin_mask

# ==========================================================
# دالة لتضبيب المشاهد المكشوفة وغير اللائقة
# ==========================================================
def blur_inappropriate_scenes(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(3))
    height = int(cap.get(4))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # كشف الأشخاص باستخدام YOLO
        results = model(frame)
        person_boxes = []
        for result in results:
            for box in result.boxes:
                cls = int(box.cls[0])
                label = model.names[cls].lower()
                if "person" in label:
                    person_boxes.append(box.xyxy[0].cpu().numpy().astype(int))

        # كشف الجلد
        skin_mask = detect_skin(frame)
        skin_blur = cv2.bitwise_and(frame, frame, mask=skin_mask)

        # تضبيب مناطق الجلد داخل كل شخص
        for (x1, y1, x2, y2) in person_boxes:
            roi = frame[y1:y2, x1:x2]
            roi_skin = skin_mask[y1:y2, x1:x2]
            if np.mean(roi_skin) > 10:  # يعني في جلد واضح
                blur = cv2.GaussianBlur(roi, (99, 99), 30)
                frame[y1:y2, x1:x2] = np.where(roi_skin[:, :, None] > 0, blur, roi)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"✅ تم إنشاء الفيديو الجديد: {output_path}")

# ==========================================================
# تحليل الصوت
# ==========================================================
def analyze_audio(video_path):
    video = mp.VideoFileClip(video_path)
    audio_path = "temp_audio.wav"
    video.audio.write_audiofile(audio_path, verbose=False, logger=None)
    segments, _ = whisper_model.transcribe(audio_path)
    full_text = " ".join([segment.text for segment in segments])
    os.remove(audio_path)
    return full_text

# ==========================================================
# التنفيذ الرئيسي
# ==========================================================
audio_text = analyze_audio(video_path)

if profanity.contains_profanity(audio_text):
    print("🚫 تم اكتشاف كلمات غير لائقة — تضبيب الفيديو بالكامل.")
    blur_inappropriate_scenes(video_path)
else:
    print("✅ النص آمن — سيتم تضبيب الأجزاء المكشوفة فقط.")
    blur_inappropriate_scenes(video_path)

# ==========================================================
# عرض الفيديو الناتج تلقائيًا
# ==========================================================
print("🎬 جاري عرض الفيديو الناتج — اضغطي Q للإغلاق.")
cap = cv2.VideoCapture(output_path)

if not cap.isOpened():
    print("❌ لم يتم العثور على الفيديو الناتج.")
else:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imshow("Blurred Video Result", frame)
        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


Streaming output truncated to the last 5000 lines.

0: 384x640 1 person, 9.9ms
Speed: 2.3ms preprocess, 9.9ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 12.0ms
Speed: 2.0ms preprocess, 12.0ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 13.5ms
Speed: 2.2ms preprocess, 13.5ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 9.2ms
Speed: 2.1ms preprocess, 9.2ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 17.1ms
Speed: 2.3ms preprocess, 17.1ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 11.3ms
Speed: 3.0ms preprocess, 11.3ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 13.6ms
Speed: 2.6ms preprocess, 13.6ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 10.1ms
Speed: 2.2ms preprocess, 10.1ms inference,

DisabledFunctionError: cv2.imshow() is disabled in Colab, because it causes Jupyter sessions
to crash; see https://github.com/jupyter/notebook/issues/3935.
As a substitution, consider using
  from google.colab.patches import cv2_imshow


In [ ]:
!pip install faster-whisper


In [ ]:
!pip install faster-whisper ultralytics better-profanity moviepy torch ffmpeg-python
!apt install ffmpeg -y


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
